In [13]:
!pip install ranx

In [14]:
# !pip install sentence-transformers pandas datasets torch ranx

import torch
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer, CrossEncoder, util
from ranx import Qrels, Run, evaluate

# Check for GPU to speed up embeddings
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Running on: {device}")

Running on: cuda


In [19]:
print("Loading Models...")
embedder = SentenceTransformer("BAAI/bge-small-en-v1.5", device=device)
reranker = CrossEncoder("BAAI/bge-reranker-base", device=device)

print("Downloading MTEB LitSearchRetrieval directly via HF Datasets...")
# MTEB datasets on Hugging Face strictly use these three configuration names
corpus_data = load_dataset("mteb/LitSearchRetrieval", "corpus")
queries_data = load_dataset("mteb/LitSearchRetrieval", "queries")
qrels_data = load_dataset("mteb/LitSearchRetrieval", "qrels")

# Dynamically select the split (usually 'corpus' for corpus, and 'test' for queries/qrels)
corpus_split = list(corpus_data.keys())[0]
queries_split = list(queries_data.keys())[0]
qrels_split = list(qrels_data.keys())[0]

corpus_ds = corpus_data[corpus_split]
queries_ds = queries_data[queries_split]
qrels_ds = qrels_data[qrels_split]

# 1. Build Queries Dictionary
# MTEB standardizes queries to have '_id' and 'text' columns
queries = {str(row['_id']): row['text'] for row in queries_ds}

# 2. Build Qrels Dictionary (Ground Truth)
# MTEB standardizes qrels to have 'query-id', 'corpus-id', and 'score'
qrels_dict = {}
for row in qrels_ds:
    q_id = str(row['query-id'])
    doc_id = str(row['corpus-id'])
    score = int(row['score'])
    
    if q_id not in qrels_dict:
        qrels_dict[q_id] = {}
    qrels_dict[q_id][doc_id] = score

# --- QUICK TESTING SUBSET ---
TESTING_MODE = False 

if TESTING_MODE:
    # Grab the first 50 queries
    test_qids = list(queries.keys())[:50]
    queries = {q: queries[q] for q in test_qids}
    qrels_dict = {q: qrels_dict.get(q, {}) for q in test_qids}

# 3. Build the Corpus
corpus = {}
valid_paper_ids = set()

# Find all the documents we actually need for our queries
for q_id in queries.keys():
    valid_paper_ids.update(qrels_dict.get(q_id, {}).keys())

print("Building Corpus...")
for row in corpus_ds:
    doc_id = str(row['_id'])
    
    if TESTING_MODE and doc_id not in valid_paper_ids and len(corpus) > 5000:
        continue
        
    title = row.get('title', '') or ''
    text = row.get('text', '') or ''
    corpus[doc_id] = f"{title}. {text}".strip()

print(f"✅ Success! Loaded {len(queries)} Queries and {len(corpus)} Papers.")

# 4. Encode the Corpus
corpus_ids = list(corpus.keys())
corpus_texts = list(corpus.values())

print("Encoding Corpus with bge-small (Stage 1)...")
corpus_embeddings = embedder.encode(corpus_texts, convert_to_tensor=True, show_progress_bar=True,batch_size=128)

Loading Models...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: BAAI/bge-small-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: BAAI/bge-reranker-base
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Building Corpus...
✅ Success! Loaded 597 Queries and 64183 Papers.
Encoding Corpus with bge-small (Stage 1)...


Batches:   0%|          | 0/502 [00:00<?, ?it/s]

In [21]:
top_k_retrieval = 20

stage1_run_dict = {}
stage2_run_dict = {}

print("Evaluating Pipeline...")

for q_id, q_text in queries.items():
    # ==========================================
    # STAGE 1: Fast Vector Retrieval
    # ==========================================
    q_emb = embedder.encode(q_text, convert_to_tensor=True)
    hits = util.semantic_search(q_emb, corpus_embeddings, top_k=top_k_retrieval)[0]
    
    stage1_run_dict[q_id] = {}
    stage1_docs = []
    
    for hit in hits:
        doc_id = corpus_ids[hit['corpus_id']]
        stage1_run_dict[q_id][doc_id] = float(hit['score'])
        stage1_docs.append(doc_id)
    
    # ==========================================
    # STAGE 2: Precision Re-ranking
    # ==========================================
    cross_inp = [[q_text, corpus[doc_id]] for doc_id in stage1_docs]
    cross_scores = reranker.predict(cross_inp)
    
    stage2_run_dict[q_id] = {}
    for doc_id, cross_score in zip(stage1_docs, cross_scores):
        stage2_run_dict[q_id][doc_id] = float(cross_score)

print("Evaluation Complete!")

Evaluating Pipeline...
Evaluation Complete!


In [23]:
# Convert to ranx objects
qrels = Qrels(qrels_dict)
run_stage1 = Run(stage1_run_dict)
run_stage2 = Run(stage2_run_dict)

print("Calculating metrics...")
metrics_to_calculate = ["mrr@5", "recall@5", "recall@10"]

results_stage1 = evaluate(qrels, run_stage1, metrics_to_calculate)
results_stage2 = evaluate(qrels, run_stage2, metrics_to_calculate)

# Build Report Table
metrics_table = {
    "Model Architecture": [
        "Stage 1 Only (bge-small)", 
        "Stage 1 + Stage 2 (bge-small + bge-reranker)"
    ],
    "MRR@5": [results_stage1["mrr@5"], results_stage2["mrr@5"]],
    "Recall@5": [results_stage1["recall@5"], results_stage2["recall@5"]],
    "Recall@10": [results_stage1["recall@10"], results_stage2["recall@10"]]
}

df_results = pd.DataFrame(metrics_table)
df_results.set_index("Model Architecture", inplace=True)

display(df_results.round(4))

Calculating metrics...


,MRR@5,Recall@5,Recall@10
Model Architecture,,,
Stage 1 Only (bge-small),0.3879,0.5043,0.5786
Stage 1 + Stage 2 (bge-small + bge-reranker),0.4523,0.5553,0.6039
